In [ ]:
# Environment Sync, GPU Check & Auto-Dataset Download

import os, sys, shutil
from pathlib import Path
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

REPO_NAME = "food-classification-deep-learning"
BRANCH = "feature/kaveesha-efficientnetb0"

if 'google.colab' in sys.modules:
    print("[INFO] Running in Google Colab environment.")
    if not os.path.exists(f"/content/{REPO_NAME}"):
        !git clone -b {BRANCH} https://github.com/niRmana11/food-classification-deep-learning.git
        %cd /content/{REPO_NAME}
    else:
        %cd /content/{REPO_NAME}
        !git checkout {BRANCH}
        !git pull origin {BRANCH}

    if f"/content/{REPO_NAME}" not in sys.path:
        sys.path.insert(0, f"/content/{REPO_NAME}")

# Locked experimental protocol: seed 42 (configs/config.yaml) for weight initialization
# and any framework-level randomness. The data loader seeds its own shuffling separately.
tf.keras.utils.set_random_seed(42)

# Verify GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"[SUCCESS] GPU active: {gpus[0].name}")
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
else:
    print("[WARNING] No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU")

# Auto-download dataset if missing (~5GB, skips if already extracted)
if not os.path.exists("data/raw/food-101/images"):
    !python -m src.data.download_food101

print(f"[INFO] Class folders found: {len(os.listdir('data/raw/food-101/images'))} (expect 101)")

In [ ]:
# Instantiate Standardized Data Loaders

from src.preprocessing.data_loader import get_food101_datasets
from src.models.efficientnetb0 import (
    build_efficientnetb0, unfreeze_top_blocks, compile_model, FINETUNE_BLOCKS
)

BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

print("[INFO] Loading datasets via shared factory...")
train_ds, val_ds, test_ds = get_food101_datasets(
    data_dir="data/raw/food-101",
    splits_dir="data/splits",
    model_type="efficientnetb0",   # Pass-through: EfficientNet rescales internally
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

print("[SUCCESS] Train, Validation and Test pipelines loaded successfully.")

In [ ]:
# SMOKE TEST - validate the full training path on ~640 images before spending GPU hours.
# Set SMOKE_TEST = False and re-run this cell once it passes, then continue to Cell 3.

SMOKE_TEST = False

if SMOKE_TEST:
    print("[SMOKE] Building throwaway model (does not affect the real run)...")
    _m, _base = build_efficientnetb0()

    print("\n[SMOKE] Phase 1 - 1 epoch on 20 batches...")
    _m.fit(train_ds.take(20), validation_data=val_ds.take(5), epochs=1, verbose=1)

    print("\n[SMOKE] Phase 2 - unfreeze, RECOMPILE, 1 epoch...")
    _n = unfreeze_top_blocks(_base)
    compile_model(_m, learning_rate=1e-5)
    print(f"[SMOKE] Unfroze {_n} layers; trainable tensors = {len(_m.trainable_weights)}")
    _m.fit(train_ds.take(20), validation_data=val_ds.take(5), epochs=1, verbose=1)

    print("\n[SMOKE] Checking prediction / label alignment...")
    _p = _m.predict(test_ds.take(5), verbose=0)
    _y = np.concatenate([y.numpy() for _, y in test_ds.take(5)])
    assert len(_p) == len(_y), "Prediction/label length mismatch!"
    assert _p.shape[1] == 101, f"Expected 101 output classes, got {_p.shape[1]}"
    print(f"[SMOKE] OK - {len(_p)} predictions aligned with {len(_y)} labels.")

    del _m, _base
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)   # restore seed state for the real run
    print("\n[SMOKE] PASSED. Set SMOKE_TEST = False, re-run this cell, then go to Cell 3.")
else:
    print("[INFO] Smoke test skipped - proceeding to the full run.")

In [ ]:
# Model Instantiation & Callbacks (Phase 1: Frozen Feature Extraction)

RESULTS_DIR = Path("results/efficientnetb0")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

model, base_model = build_efficientnetb0(
    input_shape=(224, 224, 3),
    num_classes=101,
    learning_rate=0.001,
    dropout_rate=0.3
)

# Save architecture summary to model_summary.txt (required artifact)
with open(RESULTS_DIR / "model_summary.txt", "w") as f:
    model.summary(print_fn=lambda x: f.write(x + "\n"))


def make_callbacks(phase: int, min_lr: float):
    """
    Standardized callbacks, parameterised by phase.

    min_lr must differ between phases: Phase 2 STARTS at 1e-5, so a 1e-5 floor would
    make ReduceLROnPlateau a no-op during fine-tuning. Phase 1 uses 1e-5, Phase 2 1e-7.
    """
    return [
        # Halt when val_loss stops improving; roll back to the best epoch's weights
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True, verbose=1
        ),
        # Fires first (patience 2 < 5): one automatic LR rescue before EarlyStopping gives up
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.2, patience=2, min_lr=min_lr, verbose=1
        ),
        # Per-epoch history so a Colab disconnect still leaves usable training curves
        tf.keras.callbacks.CSVLogger(
            filename=str(RESULTS_DIR / f"history_phase{phase}.csv"), separator=",", append=False
        ),
        # Per-epoch best checkpoint so a disconnect costs one epoch, not the whole run
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(RESULTS_DIR / f"effnetb0_phase{phase}_best.weights.h5"),
            monitor="val_loss", save_best_only=True, save_weights_only=True, verbose=0
        ),
    ]


trainable_p1 = int(sum(int(tf.size(w)) for w in model.trainable_weights))
print(f"[INFO] Total parameters      : {model.count_params():,}")
print(f"[INFO] Phase 1 trainable     : {trainable_p1:,} ({len(model.trainable_weights)} tensors)")
print("[INFO] Callbacks configured. Ready for Phase 1 training.")

In [ ]:
# Phase 1 Training (Frozen Backbone, lr=1e-3)
# Phase 1 was completed on 23 Sep (Colab T4, 8 epochs, best val_accuracy 0.6602).
# Leave RUN_PHASE1 = False to restore that checkpoint from Drive in a later cell.
# Set True only if you need to retrain Phase 1 from scratch (~93 minutes).

RUN_PHASE1 = False
EPOCHS_PHASE1 = 8

if RUN_PHASE1:
    print(f"[INFO] Starting EfficientNetB0 Phase 1 for up to {EPOCHS_PHASE1} epochs...")
    start_p1 = time.time()
    history_p1 = model.fit(
        train_ds, validation_data=val_ds,
        epochs=EPOCHS_PHASE1, callbacks=make_callbacks(phase=1, min_lr=1e-5)
    )
    time_p1 = time.time() - start_p1
    print(f"\n[SUCCESS] Phase 1 finished in {time_p1:.1f}s ({time_p1/60:.1f} min).")
    print(f"[RESULT] Best Phase 1 val_accuracy : {max(history_p1.history['val_accuracy']):.4f}")
else:
    print("[INFO] Phase 1 skipped - restoring the 23 Sep checkpoint from Drive instead.")

In [ ]:
# Google Drive Checkpointing
# The default callbacks write into /content, which Colab deletes when the session ends.
# These write straight to Drive, so a disconnect costs one epoch instead of the whole run.

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/food101_checkpoints"
os.makedirs(DRIVE_DIR, exist_ok=True)


def make_callbacks_drive(phase: int, min_lr: float):
    """
    Identical to make_callbacks, but checkpoint and history persist to Drive.

    min_lr differs by phase: Phase 2 STARTS at 1e-5, so a 1e-5 floor would make
    ReduceLROnPlateau a no-op during fine-tuning. Phase 1 uses 1e-5, Phase 2 1e-7.
    """
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.2, patience=2, min_lr=min_lr, verbose=1),
        tf.keras.callbacks.CSVLogger(
            f"{DRIVE_DIR}/history_phase{phase}.csv", separator=",", append=False),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=f"{DRIVE_DIR}/effnetb0_phase{phase}_best.weights.h5",
            monitor="val_loss", save_best_only=True, save_weights_only=True, verbose=1),
    ]


print(f"[INFO] Checkpoints and history will persist to: {DRIVE_DIR}")

In [ ]:
# Restore Phase 1 from Drive (trained 23 Sep: 8 epochs, best val_accuracy 0.6602)

PHASE1_WEIGHTS = f"{DRIVE_DIR}/effnetb0_phase1_best.weights.h5"
assert os.path.exists(PHASE1_WEIGHTS), (
    f"Not found: {PHASE1_WEIGHTS}\n"
    "Upload effnetb0_phase1_best.weights.h5 to Drive -> MyDrive/food101_checkpoints/"
)

model.load_weights(PHASE1_WEIGHTS)   # optimizer-state warning here is expected and harmless
time_p1 = 5590.4                     # measured Phase 1 wall-clock (s); needed for metrics.json

# Sanity check: must be far above random (1/101 = 0.0099). The slice is the first
# ~13 classes alphabetically (val is unshuffled), so expect roughly 0.60-0.70.
_l, _a, _t5 = model.evaluate(val_ds.take(30), verbose=0)
print(f"[CHECK] Restored val slice: acc={_a:.4f} top5={_t5:.4f} loss={_l:.4f}")
assert _a > 0.40, "Phase 1 weights did not load correctly - stop and investigate."
print("[SUCCESS] Phase 1 restored. Skipping the 93-minute retrain.")

In [ ]:
# Phase 2: Selective Unfreezing + Fine-Tuning at 100x Reduced Learning Rate

n_unfrozen = unfreeze_top_blocks(base_model, freeze_batchnorm=True)

# CRITICAL: Keras caches the trainable-weight list at compile time. Without this
# recompile the unfreeze is silently ignored and Phase 2 trains the head only.
compile_model(model, learning_rate=1e-5)

trainable_p2 = int(sum(int(tf.size(w)) for w in model.trainable_weights))
print(f"[INFO] Unfroze {n_unfrozen} layers in {FINETUNE_BLOCKS} (BatchNorm kept frozen).")
print(f"[INFO] Phase 2 trainable: {trainable_p2:,} of {model.count_params():,}")
assert trainable_p2 > trainable_p1, "Unfreeze did not take effect - check the recompile."

EPOCHS_PHASE2 = 10

print(f"\n[INFO] Starting Phase 2 fine-tuning for up to {EPOCHS_PHASE2} epochs...")
start_p2 = time.time()

history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE2,
    callbacks=make_callbacks_drive(phase=2, min_lr=1e-7)   # <- persists to Drive
)

time_p2 = time.time() - start_p2
total_training_time = time_p1 + time_p2
print(f"\n[SUCCESS] Phase 2 finished in {time_p2:.1f}s ({time_p2/60:.1f} min).")
print(f"[SUCCESS] Total training time: {total_training_time/3600:.2f} h.")

FINAL_WEIGHTS = str(RESULTS_DIR / "efficientnetb0_best.weights.h5")
model.save_weights(FINAL_WEIGHTS)
shutil.copy(FINAL_WEIGHTS, DRIVE_DIR)
print(f"[SAVED] {FINAL_WEIGHTS} (and backed up to Drive)")

In [ ]:
# Merge Phase Histories & Plot Training Curves (report figure)

# Phase 2 history lives on Drive; Phase 1 history is committed in the repo.
shutil.copy(f"{DRIVE_DIR}/history_phase2.csv", RESULTS_DIR / "history_phase2.csv")

h1 = pd.read_csv(RESULTS_DIR / "history_phase1.csv"); h1["phase"] = 1
h2 = pd.read_csv(RESULTS_DIR / "history_phase2.csv"); h2["phase"] = 2
h2["epoch"] = h2["epoch"] + h1["epoch"].max() + 1

hist_df = pd.concat([h1, h2], ignore_index=True)
hist_df.to_csv(RESULTS_DIR / "history.csv", index=False)

boundary = h1["epoch"].max() + 0.5
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(hist_df["epoch"], hist_df["loss"], label="Training Loss", color="#2b5c8f", linewidth=2)
ax1.plot(hist_df["epoch"], hist_df["val_loss"], label="Validation Loss", color="#e28743", linewidth=2)
ax1.axvline(boundary, color="#888888", linestyle="--", linewidth=1.5, label="Fine-tuning starts")
ax1.set_title("EfficientNetB0: Loss Progression", fontsize=12, fontweight="bold")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Cross-Entropy Loss"); ax1.legend()

ax2.plot(hist_df["epoch"], hist_df["accuracy"], label="Training Accuracy", color="#2b5c8f", linewidth=2)
ax2.plot(hist_df["epoch"], hist_df["val_accuracy"], label="Validation Accuracy", color="#e28743", linewidth=2)
ax2.axvline(boundary, color="#888888", linestyle="--", linewidth=1.5, label="Fine-tuning starts")
ax2.set_title("EfficientNetB0: Accuracy Progression", fontsize=12, fontweight="bold")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_curves.png", dpi=300)
plt.show()
print(f"[SAVED] training_curves.png and history.csv ({len(hist_df)} epochs total)")

In [ ]:
# Final Test Evaluation, Computational Profiling & metrics.json / config.yaml

print("[INFO] Evaluating best restored weights on the UNSEEN test set (25,250 images)...")
test_loss, test_acc, test_top5 = model.evaluate(test_ds, verbose=1)

# Latency (batched) - same method as the Custom CNN baseline, for comparability
latencies = []
for imgs, _ in test_ds.take(16):
    t0 = time.time()
    _ = model(imgs, training=False)
    latencies.append((time.time() - t0) / len(imgs) * 1000)
avg_latency = float(np.mean(latencies))

# Latency (batch=1) - deployment-realistic, warm-up discarded
single = next(iter(test_ds.unbatch().batch(1)))[0]
for _ in range(20):
    _ = model(single, training=False)
t0 = time.time()
for _ in range(100):
    _ = model(single, training=False)
latency_b1 = (time.time() - t0) / 100 * 1000

model_size_mb = os.path.getsize(FINAL_WEIGHTS) / (1024 ** 2)
try:
    peak_vram_mb = tf.config.experimental.get_memory_info('GPU:0')['peak'] / (1024 ** 2)
except Exception:
    peak_vram_mb = None

metrics_data = {
    # keys identical to results/custom_cnn/metrics.json so the aggregator parses uniformly
    "model_name": "EfficientNetB0",
    "total_parameters": int(model.count_params()),
    "trainable_parameters": trainable_p2,
    "training_time_seconds": round(total_training_time, 2),
    "inference_latency_ms_per_image": round(avg_latency, 2),
    "final_val_loss": round(float(hist_df["val_loss"].min()), 4),
    "final_val_accuracy": round(float(hist_df["val_accuracy"].max()), 4),
    "final_test_loss": round(float(test_loss), 4),
    "final_test_accuracy": round(float(test_acc), 4),
    # two-phase transfer-learning extensions
    "trainable_parameters_phase1": trainable_p1,
    "phase1_training_time_seconds": round(time_p1, 2),
    "phase2_training_time_seconds": round(time_p2, 2),
    "epochs_phase1": int(len(h1)),
    "epochs_phase2": int(len(h2)),
    "final_val_top5_accuracy": round(float(hist_df["val_top_5_accuracy"].max()), 4),
    "final_test_top5_accuracy": round(float(test_top5), 4),
    "inference_latency_ms_batch1": round(float(latency_b1), 2),
    "model_size_mb": round(model_size_mb, 2),
    "peak_gpu_memory_mb": round(peak_vram_mb, 2) if peak_vram_mb else None,
    "unfrozen_blocks": list(FINETUNE_BLOCKS),
    "batchnorm_frozen_during_finetuning": True,
}
with open(RESULTS_DIR / "metrics.json", "w") as f:
    json.dump(metrics_data, f, indent=2)

import yaml
run_config = {
    "model_name": "EfficientNetB0",
    "seed": 42,
    "image_size": list(IMAGE_SIZE),
    "batch_size": BATCH_SIZE,
    "preprocessing": "efficientnet.preprocess_input (pass-through, expects [0,255])",
    "augmentation": "train split only: RandomFlip(horizontal), RandomRotation(0.05), RandomZoom(0.10)",
    "optimizer": "adam",
    "loss": "sparse_categorical_crossentropy",
    "phase1": {"epochs_max": EPOCHS_PHASE1, "learning_rate": 0.001, "backbone": "frozen"},
    "phase2": {"epochs_max": EPOCHS_PHASE2, "learning_rate": 1e-05,
               "unfrozen": list(FINETUNE_BLOCKS), "batchnorm": "frozen"},
    "early_stopping": {"monitor": "val_loss", "patience": 5, "restore_best_weights": True},
    "reduce_lr": {"monitor": "val_loss", "factor": 0.2, "patience": 2},
    "hardware": "Google Colab NVIDIA T4 GPU",
}
with open(RESULTS_DIR / "config.yaml", "w") as f:
    yaml.safe_dump(run_config, f, sort_keys=False)

print("\nEFFICIENTNETB0 FINAL EVALUATION SUMMARY")
for k, v in metrics_data.items():
    print(f"{k:<38}: {v}")

In [ ]:
# Confusion Matrix, Classification Report & Error Analysis

from sklearn.metrics import classification_report, confusion_matrix

print("[INFO] Computing test-set predictions (single ordered pass)...")
# test_ds is deterministic (is_training=False, no shuffle), so predictions and labels
# stay aligned. One pass over the test set; everything below derives from it.
y_pred_probs = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
assert len(y_true) == len(y_pred), "Prediction/label length mismatch!"
print(f"[INFO] Aligned {len(y_true)} test predictions.")

with open("data/splits/classes.txt") as f:
    class_names = [line.strip() for line in f if line.strip()]

report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
with open(RESULTS_DIR / "classification_report.json", "w") as f:
    json.dump(report, f, indent=2)
print("[SAVED] classification_report.json")

cm = confusion_matrix(y_true, y_pred, normalize='true')
plt.figure(figsize=(24, 20))
sns.heatmap(cm, cmap="Blues", xticklabels=False, yticklabels=False, cbar=True, rasterized=True)
plt.title("EfficientNetB0: Normalized 101-Class Confusion Matrix (Test Set)",
          fontsize=16, fontweight="bold", pad=15)
plt.xlabel("Predicted Class (101 Categories)", fontsize=13)
plt.ylabel("True Class (101 Categories)", fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=300)
plt.show()
print("[SAVED] confusion_matrix.png")

# Section 8 raw material: most-confused pairs and worst-recall classes
off_diag = cm.copy()
np.fill_diagonal(off_diag, 0)
pairs = []
for idx in np.argsort(off_diag, axis=None)[::-1][:10]:
    i, j = np.unravel_index(idx, off_diag.shape)
    pairs.append({"true_class": class_names[i], "predicted_as": class_names[j],
                  "confusion_rate": round(float(off_diag[i, j]), 4)})

worst = sorted([(c, report[c]["recall"], report[c]["f1-score"]) for c in class_names],
               key=lambda r: r[1])[:10]

error_analysis = {
    "top_confused_pairs": pairs,
    "worst_recall_classes": [{"class": c, "recall": round(r, 4), "f1": round(f, 4)}
                             for c, r, f in worst],
    "macro_precision": round(float(report["macro avg"]["precision"]), 4),
    "macro_recall": round(float(report["macro avg"]["recall"]), 4),
    "macro_f1": round(float(report["macro avg"]["f1-score"]), 4),
    "weighted_f1": round(float(report["weighted avg"]["f1-score"]), 4),
}
with open(RESULTS_DIR / "error_analysis.json", "w") as f:
    json.dump(error_analysis, f, indent=2)

print("\nTOP 10 CONFUSED CLASS PAIRS")
for p in pairs:
    print(f"  {p['true_class']:<28} -> {p['predicted_as']:<28} {p['confusion_rate']:.3f}")
print("\nWORST 10 CLASSES BY RECALL")
for c, r, f1 in worst:
    print(f"  {c:<28} recall={r:.3f}  f1={f1:.3f}")
print(f"\nMacro F1: {error_analysis['macro_f1']}   Weighted F1: {error_analysis['weighted_f1']}")

In [ ]:
# Package Artifacts: copy to Drive and download a zip

shutil.make_archive("/content/efficientnetb0_results", "zip", RESULTS_DIR)
shutil.copy("/content/efficientnetb0_results.zip", DRIVE_DIR)

print("ARTIFACTS IN", RESULTS_DIR)
for p in sorted(RESULTS_DIR.iterdir()):
    print(f"  {p.name:<42} {p.stat().st_size/1024:>10.1f} KB")

required = ["config.yaml", "history.csv", "model_summary.txt", "training_curves.png",
            "confusion_matrix.png", "classification_report.json", "metrics.json"]
missing = [r for r in required if not (RESULTS_DIR / r).exists()]
print("\n[CHECK] Required artifacts:", "ALL 7 PRESENT" if not missing else f"MISSING {missing}")

from google.colab import files
files.download("/content/efficientnetb0_results.zip")